# Open the FLARE sample
Developed by BRIDGE Research and Policy Foundation and ARTH GEO Intelligence.
Install `requirements.txt`, extract a scientific archive under `data/`, and run each cell.
Set `FLARE_DATA_DIR` or edit DATA below to use an existing directory.
This package contains 25 scenes for July 16, 2026, at two separate time intervals.
Data and examples: CC BY-NC-ND 4.0. See the repository LICENSE.


In [ ]:
from pathlib import Path
import os, sys
REPO = Path.cwd() if (Path.cwd() / 'examples').exists() else Path.cwd().parent
sys.path.insert(0, str(REPO / 'examples'))
from flare_sample import data_root, scenes, open_scene, false_rgb, nearest_pixel, interactive_map
DATA = data_root(os.environ.get('FLARE_DATA_DIR', str(REPO / 'data')))
FORMAT = 'zarr'  # Change to 'netcdf' if you downloaded that format.
TAGS = scenes(DATA, FORMAT)
OUTPUT = REPO / 'outputs'
OUTPUT.mkdir(exist_ok=True)
print(f'{len(TAGS)} scenes; {TAGS[0]} through {TAGS[-1]} (with a gap). All times UTC.')


In [ ]:
ds = open_scene(DATA, TAGS[0], FORMAT)
display(ds)
for key, value in ds.attrs.items():
    print(key, ':', value)


## Inspect the actual grid and variables
Latitude and longitude are two-dimensional and share the data grid.
All products in this sample have already been aligned to the v7.14 native ROI.
Do not apply the older v5.12 20-pixel crop offset to these files.
Valid score zero and categorical zero are retained. Floating NaN means unavailable.


In [ ]:
import numpy as np
for name in ds.data_vars:
    v = ds[name]
    print(name, v.shape, v.dtype, v.attrs.get('units', 'units not supplied'))
assert ds.lat.shape == ds.lon.shape == ds.FireScore.shape == (545, 1425)
print('Latitude:', float(ds.lat.min()), float(ds.lat.max()))
print('Longitude:', float(ds.lon.min()), float(ds.lon.max()))
print('UTC scene:', ds.time.values)
print('Valid score range:', float(ds.FireScore.min()), float(ds.FireScore.max()))


## Compare scientific formats (optional)
If both formats were downloaded, check every variable in the selected scene.
Otherwise this cell explains which download is missing.


In [ ]:
if (DATA/'Zarr').exists() and (DATA/'NETCDF').exists():
    with open_scene(DATA,TAGS[0],'zarr') as a, open_scene(DATA,TAGS[0],'netcdf') as b:
        assert set(a.variables) == set(b.variables)
        for name in a.variables:
            np.testing.assert_array_equal(a[name].values, b[name].values)
    print('All variables and coordinates match exactly.')
else:
    print('Download both Zarr and NetCDF to run the cross-format comparison.')
ds.close()
